# 01. Данные и две шкалы измерения (методы 1–3)

**Ветка:** `report-20pp`  
**Роль:** теория (блоки A–D) + уровни в общий год, среднегодовой темп, два окна для статей.

**Расшифровки:**
- **GERD** — валовые внутренние расходы на НИОКР (*Gross Domestic Expenditure on R&D*).
- **BERD** — расходы на НИОКР, выполненные бизнесом (*Business Enterprise R&D*, PERFORMED).
- **CAGR** — среднегодовой темп прироста (*compound annual growth rate*).
- **ППС** — паритет покупательной способности.


## Теория (кратко, для отчёта)

### A. Зачем сравнивать две шкалы
Национальная статистика науки публикует и **интенсивность** (например, GERD в процентах ВВП), и **объём** (абсолютные расходы, число статей). Это разные знаменатели, а не две «модели развития».

- OECD. *Frascati Manual 2015*.
- OECD/Eurostat. *Oslo Manual 2018*.

### B. Цепочка «наука → внедрение»
Линейная схема удобна как оглавление отчёта. В литературе она давно описана как упрощение: есть обратные связи и обучение на производстве (Kline & Rosenberg, 1986).

### C. Национальные инновационные системы
Страны различаются организацией науки, фирм и государства. Здесь сравниваются **измеримые индикаторы**, а не «какая система лучше» (Freeman, 1987; Lundvall, 1992; Nelson, 1993).

### D. Догоняющее развитие и масштаб
Поздний догоняющий может быстро наращивать **объём** при меньшей **интенсивности** (Abramovitz, 1986; Gerschenkron, 1962; Lee, 2013). Это рамка для чтения: Китай выше по статьям и абсолютному GERD в млн долл. по ППС; США выше по GERD% ВВП и исследователям на миллион жителей.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
PANEL_PATH = ROOT / "data_reviewed" / "core_panel_reviewed.csv"
TABLES = ROOT / "data_reviewed" / "tables_reviewed"
RAW = ROOT / "data" / "raw"
OUT_FIG = ROOT / "notebooks" / "figures"
OUT_TAB = ROOT / "notebooks" / "tables"
OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_TAB.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(PANEL_PATH)
panel = panel[(panel["year"] >= 2010) & (panel["year"] <= 2024)].copy()
US_CN = panel[panel["country_iso3"].isin(["USA", "CHN"])].copy()

def cagr(start: float, end: float, n_years: int) -> float:
    """Compound annual growth rate; n_years = end_year - start_year."""
    if start is None or end is None or np.isnan(start) or np.isnan(end) or start <= 0 or n_years <= 0:
        return np.nan
    return (end / start) ** (1 / n_years) - 1

print("panel rows", len(panel), "US+CN rows", len(US_CN))
print("countries", sorted(panel["country_iso3"].unique()))


panel rows 120 US+CN rows 30
countries ['CHN', 'DEU', 'FRA', 'GBR', 'ISR', 'JPN', 'KOR', 'USA']


## Метод 1. Уровни в общий год

Для каждого показателя берём последний год, в котором есть значения **и** у США, **и** у Китая (joint year). Рядом с цифрой всегда пишем год.


In [2]:
dual = pd.read_csv(TABLES / "dual_scale_with_absolutes.csv")
display_cols = [
    "var", "scale_family", "joint_year", "USA", "CHN",
    "CHN_USA_ratio", "gap_sign_level", "cagr_t0", "cagr_t1", "CAGR_USA", "CAGR_CHN", "note",
]
show = dual[display_cols].copy()
show.to_csv(OUT_TAB / "01_dual_scale_with_absolutes.csv", index=False)
show


,var,scale_family,joint_year,USA,CHN,CHN_USA_ratio,gap_sign_level,cagr_t0,cagr_t1,CAGR_USA,CAGR_CHN,note
0,gerd_pct_gdp,intensity,2023,3.447160e+00,2.577290e+00,0.747656,USA_higher,2010.0,2023.0,0.018552,0.033251,NaN
1,berd_pct_gdp,intensity,2024,2.667761e+00,2.091171e+00,0.783867,USA_higher,2010.0,2024.0,0.026341,0.038225,NaN
2,researchers_per_million,intensity,2022,4.937494e+03,1.849237e+03,0.374529,USA_higher,2010.0,2017.0,0.014152,0.046535,joint year latest mutual; CAGR window 2010-201...
3,scopus_articles,volume,2024,4.398920e+05,1.078580e+06,2.451920,CHN_higher,2010.0,2024.0,0.005495,0.093689,2014-2024 from NSF Indicators 2026 Fig.29; pre...
4,patents_resident,volume,2021,2.622440e+05,1.426644e+06,5.440140,CHN_higher,2010.0,2021.0,0.007339,0.154746,NaN
5,patents_total_office,volume,2021,5.914730e+05,1.585663e+06,2.680871,CHN_higher,2010.0,2021.0,0.017215,0.135685,NaN
6,mva_pct_gdp,share,2021,1.052503e+01,2.662193e+01,2.529392,CHN_higher,2010.0,2021.0,-0.011146,-0.013962,NaN
7,hitech_export_share,share,2024,2.432298e+01,2.627774e+01,1.080367,CHN_higher,2010.0,2024.0,0.005200,-0.014303,NaN
8,semi_exports_hs8542,volume_nominal,2023,4.355698e+10,1.363350e+11,3.130039,CHN_higher,NaN,NaN,NaN,NaN,Comtrade HS8542 from exports.csv (clean is USA...
9,scopus_articles_CAGR_only,volume,2021,4.723750e+05,7.615350e+05,1.612141,CHN_higher,2010.0,2021.0,0.013542,0.085824,dual-window; endpoint 2021; peak USA 2021=4723...


## Метод 2. Среднегодовой темп на общем окне (CAGR)

Формула: \(\mathrm{CAGR} = (y_{t_1}/y_{t_0})^{1/(t_1-t_0)} - 1\). Окно \((t_0, t_1)\) пишется явно для каждого ряда.


In [3]:
cagr_tbl = pd.read_csv(TABLES / "descriptive_cagr_common_window.csv")
cagr_tbl.to_csv(OUT_TAB / "01_cagr_common_window.csv", index=False)
cagr_tbl.head(20)


,var,country,t0,t1,v0,v1,CAGR_balanced
0,gerd_pct_gdp,USA,2010,2023,2.714430,3.447160,0.018552
1,gerd_pct_gdp,CHN,2010,2023,1.684560,2.577290,0.033251
2,gerd_pct_gdp,KOR,2010,2023,3.179130,4.943520,0.034542
3,gerd_pct_gdp,JPN,2010,2023,3.104950,3.441250,0.007942
4,gerd_pct_gdp,DEU,2010,2023,2.676320,3.153900,0.012711
5,gerd_pct_gdp,GBR,2010,2023,1.638870,2.676140,0.038441
6,gerd_pct_gdp,ISR,2010,2023,3.842100,6.345890,0.039354
7,gerd_pct_gdp,FRA,2010,2023,2.177710,2.178320,0.000022
8,berd_pct_gdp,USA,2010,2023,1.853795,2.657845,0.028102
9,berd_pct_gdp,CHN,2010,2023,1.236833,2.002840,0.037774


## Метод 3. Два окна для статей и год пересечения

- CAGR статей считается **дважды**: 2010–2021 (пик США) и 2010–2024 (полный актуальный горизонт).
- Пересечение объёма статей: последний год, когда США ≥ Китая — **2016**; первый год, когда Китай > США — **2017**. Формулировка «около 2020» **неверна**.
- Ряд 2014–2024 для США/Китая: **NSF Indicators 2026**, Figure 29 (Scopus, дробный подсчёт). К 2024 Китай ≈2.45× США по объёму.


In [4]:
arts = US_CN[["country_iso3", "year", "scopus_articles"]].dropna().sort_values(["year", "country_iso3"])
wide = arts.pivot(index="year", columns="country_iso3", values="scopus_articles").dropna()
wide["CHN_gt_USA"] = wide["CHN"] > wide["USA"]
crossover_years = wide.loc[wide["CHN_gt_USA"]].index.min()
last_usa_lead = wide.loc[~wide["CHN_gt_USA"]].index.max()
print("last year USA >= CHN:", int(last_usa_lead))
print("first year CHN > USA:", int(crossover_years))
if 2024 in wide.index:
    print("2024 USA/CHN:", float(wide.loc[2024, "USA"]), float(wide.loc[2024, "CHN"]),
          "ratio", float(wide.loc[2024, "CHN"] / wide.loc[2024, "USA"]))

dual_art = pd.read_csv(TABLES / "articles_cagr_dual_window.csv")
dual_art.to_csv(OUT_TAB / "01_articles_cagr_dual_window.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
for iso, color in [("USA", "#1f77b4"), ("CHN", "#d62728")]:
    s = arts[arts["country_iso3"] == iso]
    ax.plot(s["year"], s["scopus_articles"], marker="o", label=iso, color=color)
ax.axvline(2016.5, color="gray", linestyle="--", linewidth=1, label="пересечение 2016→2017")
ax.set_title("S&E статьи (NSF/Scopus): США и Китай, 2010–2024")
ax.set_xlabel("Год")
ax.set_ylabel("Число статей (дробный подсчёт)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_FIG / "01_articles_crossover.png", dpi=150)
plt.show()
dual_art


last year USA >= CHN: 2016
first year CHN > USA: 2017
2024 USA/CHN: 439892.0 1078580.0 ratio 2.4519200167313797


C:\Users\danii\AppData\Local\Temp\ipykernel_9800\3030860030.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,var,country,window_t0,window_t1,v0,v1,CAGR,peak_year,peak_value,note
0,scopus_articles,USA,2010,2021,407405.92,472375.0,0.013542,2021.0,472375.0,dual-window; 2014-2024 NSF Indicators 2026 Fig...
1,scopus_articles,USA,2010,2024,407405.92,439892.0,0.005495,2021.0,472375.0,dual-window; 2014-2024 NSF Indicators 2026 Fig...
2,scopus_articles,CHN,2010,2021,307848.30,761535.0,0.085824,NaN,NaN,dual-window; 2014-2024 NSF Indicators 2026 Fig...
3,scopus_articles,CHN,2010,2024,307848.30,1078580.0,0.093689,NaN,NaN,dual-window; 2014-2024 NSF Indicators 2026 Fig...


## Краткий вывод для отчёта (метод 1–3)

По таблице dual-scale при выбранных шкалах: США выше по **интенсивности** (GERD%, BERD%, исследователи на млн), Китай выше по **объёму/доле** (статьи, патенты, MVA%, hitech%, абсолютный GERD PPP, оценка численности исследователей). Это арифметика шкал, а не вердикт «победитель». Пересечение статей — 2016–2017; к 2024 разрыв по объёму статей вырос примерно до 2.45× (NSF 2026).
